# Overfitting-spaces immutable launcher
This notebook only checks out a pinned commit and invokes the repository runner.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL = "__REPO_URL__"
GIT_COMMIT = "__GIT_COMMIT__"
CONFIG_REL = "__CONFIG_REL__"
RUNTIME = Path("/tmp/overfitting-spaces-runtime")
SOURCE = RUNTIME / "source"
OUTPUT = Path("/kaggle/working/overfitting-results")
assert len(GIT_COMMIT) == 40 and all(c in "0123456789abcdef" for c in GIT_COMMIT)
assert not SOURCE.exists(), f"fresh session required: {SOURCE}"
RUNTIME.mkdir(parents=True, exist_ok=True); OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
env = os.environ.copy(); env.update({"GIT_TERMINAL_PROMPT": "0", "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1", "PIP_DISABLE_PIP_VERSION_CHECK": "1"})
subprocess.run(["git", "clone", REPO_URL, str(SOURCE)], check=True, env=env)
subprocess.run(["git", "-C", str(SOURCE), "checkout", "--detach", GIT_COMMIT], check=True, env=env)
assert subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip() == GIT_COMMIT
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(SOURCE / "requirements-kaggle.txt")], check=True, env=env)


In [ ]:
cmd = [sys.executable, "-m", "overfitting_spaces.runner", "--config", str(SOURCE / CONFIG_REL), "--output-root", str(OUTPUT)]
result = subprocess.run(cmd, cwd=str(SOURCE), env={**env, "PYTHONPATH": str(SOURCE / "src"), "OVERFIT_GIT_SHA": GIT_COMMIT}, check=False)
if result.returncode: raise RuntimeError(f"runner failed ({result.returncode}); diagnostics remain under {OUTPUT}")
